# Task 4.2

## Understanding the effects of Non-Poissonian spiking


Ie redoing task 1 with different shapes of the emmission distribution.

The shapes used are usually 1, 3, and 5 with shape = 1 the same as poisson, and 3 adn 5 I think being sub poisson

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import models
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import datavis2 as dv
from scipy.ndimage import gaussian_laplace as gl
from scipy.ndimage import gaussian_filter
from scipy.ndimage import gaussian_filter1d
import task1_classifiers.firstclassifier as fc


plt.rcParams['font.family'] = ['cmr10', 'Times New Roman', 'STIXGeneral']


## Task 1.1.

 Ramp Model

 Pretty much exact same raster plots for all shapes

In [ ]:
dv.rampRasterPlot2([1.0], [0.25])
dv.rampRasterPlot3([1.0], [0.25])

dv.rampRasterPlot2([0.25, 1, 3], [0.1, 0.25, 0.75])
dv.rampRasterPlot3([0.25, 1, 3], [0.1, 0.25, 0.75])

#dv.rampWalkPlot([0.25, 1, 3], [0.1, 0.25, 0.75])


Step Model

Again, same for all shapes

In [ ]:
dv.stepRasterPlot2([500], [6])
dv.stepRasterPlot3([500], [6])

dv.stepRasterPlot2([250, 500, 750], [0.5, 6, 60])
dv.stepRasterPlot3([250, 500, 750], [0.5, 6, 60])

#dv.stepWalkPlot([250, 500, 750], [0.5, 6, 60])

## Task 1.2.

Once again, almost exactly the same, except for small times where the ramp model differs. Not entirely sure why

In [ ]:
#Singles for tests
dv.psth_grid_with_shapes(models.RampModel, [1], [0.2], "β", "σ",
          n_trials=200, bin_width=20, smooth_ms=20)
dv.psth_grid_with_shapes(models.StepModel, [500], [6], "m", "r",
          n_trials=200, bin_width=20, smooth_ms=20)

# A) basic PSTH grids
dv.psth_grid_with_shapes(models.RampModel, [0.5,1,3], [0.04,0.2,0.5], "β", "σ",
          n_trials=200, bin_width=20, smooth_ms=20)
dv.psth_grid_with_shapes(models.StepModel, [250,500,750], [0.5,6,60], "m", "r",
          n_trials=200, bin_width=20, smooth_ms=20)
plt.show()

In [ ]:
# Not needed for task 4.2

# analyse the psth plots
'''N_TRIALS = 5000
T_BINS = 1000
for m in [200]:
    for r in [0.5]:
        step_model = models.StepModel(m=m, r=r)
        spikes, *_ = step_model.simulate(Ntrials=N_TRIALS, T=T_BINS)

        t = np.linspace(0, 1, T_BINS)
        dv.analyse_psth(spikes, t, f"Step PSTH  m={m}, r={r}")


for beta in [0]:
    for sigma in [4]:
        ramp_model = models.RampModel(beta=beta, sigma=sigma)
        spikes, *_ = ramp_model.simulate(Ntrials=N_TRIALS, T=T_BINS)

        t = np.linspace(0, 1, T_BINS)
        dv.analyse_psth(spikes, t, f"Ramp PSTH β={beta}, σ={sigma}")'''

In [ ]:
# C) trial-count dependence
'''Ns, errs = dv.psth_variability(models.RampModel, dict(beta=1, sigma=.15))
plt.figure(); plt.loglog(Ns, errs, marker='o');
plt.xlabel("number of trials"); plt.ylabel("avg PSTH SD (Hz)");
plt.title("Ramp β=1 σ=.15  –  PSTH variability"); plt.grid(True, which='both'); plt.show()

# D) Overlay to hunt for matching PSTHs
dv.overlay_psth_comparison(dict(beta=0.8, sigma=.2),
                        dict(m=400, r=20),
                        n_trials=400, bin_width=20, smooth_ms=20)
plt.show()'''

## Task 1.3.

In [ ]:
# generate a dataset
T = 1000
datasize = 10
trials = 400
bw=20

# generate a random set of ramp parameters
beta = np.random.uniform(0, 4, datasize)
ln_sig = np.random.uniform(np.log(0.04), np.log(20), datasize)
ramp_params = np.column_stack((beta, np.exp(ln_sig)))

# generate a random set of step parameters
r = np.random.uniform(0.5, 100, datasize)
m = np.random.uniform(T/4, 3*T/4, datasize)
step_params = np.column_stack((m, r))

x_init = np.random.uniform(0, 0.5, datasize*2)

In [ ]:
# generate an array of fano factors time series arrays
sfac = 100
bw = 20 # bin width for the mean and variances to be calculated along
fanos = np.zeros((40, int(1000/bw)))

fig, ax1 = plt.subplots(figsize=(12,6))
fig, ax2 = plt.subplots(figsize=(12,6))

for i in range(datasize):
    _, fanos[i, :] = dv.fanoFactor(models.RampModel(beta=ramp_params[i][0],
                                                    sigma=ramp_params[i][1], 
                                                    x0=x_init[i]),
                                bin_width=bw,
                                smooth_ms=sfac,
                                ax=ax1,
                                plot=True
                            )

for j in range(datasize):              
    times, fanos[j+datasize, :] = dv.fanoFactor(models.StepModel(m=step_params[j][0],
                                                                r=step_params[j][1],
                                                                x0=x_init[j+datasize]),
                                    bin_width=bw,
                                    smooth_ms=sfac,
                                    ax=ax2,
                                    plot=True
                            )


In [ ]:
# clip start and end of arrays
n = len(fanos[0])
clip_frac = 0.1 # fraction of array to remove from start and end
fanos = np.array(fanos[:, int(np.floor(n*clip_frac)) : int(np.ceil(n*(1-clip_frac)))])

times = dv.fanoFactor(models.StepModel(m=step_params[0][0],r=step_params[0][1],x0=x_init[0]), bin_width=bw)[0]
times = np.array(times[int(np.floor(n*clip_frac)) : int(np.ceil(n*(1-clip_frac)))])

In [ ]:
# generate a dataset
T = 1000
datasize = 10
trials = 400
bw=20

# generate a random set of ramp parameters
beta = np.linspace(0, 4, datasize)
ln_sig = np.linspace(np.log(0.04), np.log(20), datasize)
ramp_params = np.column_stack((beta, np.exp(ln_sig)))

# generate a random set of step parameters
r = np.linspace(0.5, 100, datasize)
m = np.linspace(T/4, 3*T/4, datasize)
step_params = np.column_stack((m, r))

x_init = np.random.uniform(0, 0.5, datasize*2)

# generate an array of fano factors time series arrays
sfac = 100
bw = 20 # bin width for the mean and variances to be calculated along
fanos = np.zeros((40, int(1000/bw)))

fig, ax1 = plt.subplots(figsize=(12,6))
fig, ax2 = plt.subplots(figsize=(12,6))

for i in range(datasize):
    _, fanos[i, :] = dv.fanoFactor(models.RampModel(beta=ramp_params[i][0],
                                                    sigma=ramp_params[i][1], 
                                                    x0=x_init[i]),
                                bin_width=bw,
                                smooth_ms=sfac,
                                ax=ax1,
                                plot=True
                            )

for j in range(datasize):              
    times, fanos[j+datasize, :] = dv.fanoFactor(models.StepModel(m=step_params[j][0],
                                                                r=step_params[j][1],
                                                                x0=x_init[j+datasize]),
                                    bin_width=bw,
                                    smooth_ms=sfac,
                                    ax=ax2,
                                    plot=True
                            )

In [ ]:
# generate a dataset
#Using linspace

T = 1000
datasize = 5
trials = 200
bw=20

# generate a random set of ramp parameters
betas = np.linspace(0, 4, datasize)
ln_sigs = np.linspace(np.log(0.04), np.log(20), datasize)
sigs = np.exp(ln_sigs)
ramp_params = np.column_stack((betas, np.exp(ln_sigs)))


x_init = np.random.uniform(0, 0.5, datasize*2)

# generate an array of fano factors time series arrays
sfac = 100
bw = 20 # bin width for the mean and variances to be calculated along
fanos = np.zeros((40, int(1000/bw)))

fig, ax1 = plt.subplots(figsize=(12,6))


shapes = [1, 3, 5]
colors = ['r', 'g', 'b']
shape_colors = dict(zip(shapes, colors))

for b, beta in enumerate(betas):
    for s, sigma in enumerate(sigs):
        for shape in shapes:
            model = models.RampModel(beta=beta, sigma=sigma, isi_gamma_shape=shape)
            dv.fanoFactor2(model, 
                           bin_width=bw,
                            smooth_ms=sfac,
                            plot=True, 
                            ax=ax1, 
                            color=shape_colors[shape])

# legend just for shapes
legend_handles = [mpatches.Patch(color=shape_colors[s], label=f"shape={s}") for s in shapes]
ax1.legend(handles=legend_handles, title="isi_gamma_shape")


In [ ]:
# generate a dataset
#Using ranom

T = 1000
datasize = 5
trials = 200
bw=20

# generate a random set of ramp parameters
betas = np.random.uniform(0, 4, datasize)
ln_sigs = np.random.uniform(np.log(0.04), np.log(20), datasize)
sigs = np.exp(ln_sigs)
ramp_params = np.column_stack((betas, np.exp(ln_sigs)))


x_init = np.random.uniform(0, 0.5, datasize*2)

# generate an array of fano factors time series arrays
sfac = 100
bw = 20 # bin width for the mean and variances to be calculated along
fanos = np.zeros((40, int(1000/bw)))

fig, ax1 = plt.subplots(figsize=(12,6))


shapes = [1, 3, 5]
colors = ['r', 'g', 'b']
shape_colors = dict(zip(shapes, colors))

for b, beta in enumerate(betas):
    for s, sigma in enumerate(sigs):
        for shape in shapes:
            model = models.RampModel(beta=beta, sigma=sigma, isi_gamma_shape=shape)
            dv.fanoFactor2(model, 
                           bin_width=bw,
                            smooth_ms=sfac,
                            plot=True, 
                            ax=ax1, 
                            color=shape_colors[shape])

# legend just for shapes
legend_handles = [mpatches.Patch(color=shape_colors[s], label=f"shape={s}") for s in shapes]
ax1.legend(handles=legend_handles, title="isi_gamma_shape")


In [ ]:
datasize = 5

fig, ax2 = plt.subplots(figsize=(12,6))

# generate a random set of step parameters
r_list = np.linspace(0.5, 100, datasize)
m_list = np.linspace(T/4, 3*T/4, datasize)
step_params = np.column_stack((m_list, r_list))

for m in m_list:
    for r in r_list:
        for shape in shapes:
            model = models.StepModel(m=m, r=r, isi_gamma_shape=shape)
            dv.fanoFactor2(model, 
                           bin_width=bw,
                            smooth_ms=sfac,
                            plot=True, 
                            ax=ax2, 
                            color=shape_colors[shape])

# legend just for shapes
legend_handles = [mpatches.Patch(color=shape_colors[s], label=f"shape={s}") for s in shapes]
ax2.legend(handles=legend_handles, title="isi_gamma_shape")

In [ ]:
datasize = 5

fig, ax2 = plt.subplots(figsize=(12,6))

# generate a random set of step parameters
r_list = np.random.uniform(0.5, 100, datasize)
m_list = np.random.uniform(T/4, 3*T/4, datasize)
step_params = np.column_stack((m_list, r_list))

for m in m_list:
    for r in r_list:
        for shape in shapes:
            model = models.StepModel(m=m, r=r, isi_gamma_shape=shape)
            dv.fanoFactor2(model, 
                           bin_width=bw,
                            smooth_ms=sfac,
                            plot=True, 
                            ax=ax2, 
                            color=shape_colors[shape])

# legend just for shapes
legend_handles = [mpatches.Patch(color=shape_colors[s], label=f"shape={s}") for s in shapes]
ax2.legend(handles=legend_handles, title="isi_gamma_shape")

## Task 1.4.

In [ ]:
# plot numerical derivatives of fano factor wrt time
fanos_smoothed = np.array([gaussian_filter1d(fanos[i], sigma=3) for i in range(datasize*2)])
gradients = [np.gradient(fanos_smoothed[i], times) for i in range(datasize*2)]


fig, ax3 = plt.subplots(figsize=(6,3))
fig, ax4 = plt.subplots(figsize=(6,3))
fig, ax34 = plt.subplots(figsize=(6,3))

for i in range(datasize):
    ax3.plot(times, gradients[i], label='ramp ' + str(ramp_params[i]))
    ax34.plot(times, gradients[i], c='red')

for j in range(datasize):
    ax4.plot(times, gradients[j+datasize], label='step ' + str(step_params[j]))
    ax34.plot(times, gradients[j+datasize], c='blue')


ax3.legend()
ax4.legend()

ax3.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax4.legend(loc='center left', bbox_to_anchor=(1, 0.5))

fig.tight_layout()
plt.show()



In [ ]:
# convolve the fano factor array with Laplacian of Gaussian kernel
LoG_responses = [gl(fanos[i, :], sigma=8) for i in range(datasize*2)]


fig, ax5 = plt.subplots(figsize=(6,3))
fig, ax6 = plt.subplots(figsize=(6,3))
fig, ax56 = plt.subplots(figsize=(6,3))

for i in range(datasize):
    ax5.plot(times, LoG_responses[i], label='ramp '+str(ramp_params[i]))
    ax56.plot(times, LoG_responses[i], c='red')

for j in range(datasize):
    ax6.plot(times, LoG_responses[j+datasize], label='step '+str(step_params[j]))
    ax56.plot(times, LoG_responses[j+datasize], c='blue')

ax5.legend()
ax6.legend()
ax5.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax6.legend(loc='center left', bbox_to_anchor=(1, 0.5))

fig.tight_layout()
plt.show()

In [ ]:
# plot traces of fano factor once convolved with laplacian kernel
laplacians = np.array([dv.laplacian(fanos_smoothed[i], plot=False)  for i in range(datasize*2)])

n = len(laplacians[0])
clip_fac = 0.1 # fraction of array to remove from start and end
laplacians = laplacians[:, int(np.floor(n*clip_fac)) : int(np.ceil(n*(1-clip_fac)))]
times = times[int(np.floor(n*clip_fac)) : int(np.ceil(n*(1-clip_fac)))]

fig, ax7 = plt.subplots(figsize=(6,3))
fig, ax8 = plt.subplots(figsize=(6,3))
fig, ax78 = plt.subplots(figsize=(6,3))

for i in range(datasize):
    ax7.plot(times, laplacians[i])
    ax78.plot(times, laplacians[i], c='red')

for j in range(datasize):
    ax8.plot(times, laplacians[j+datasize])
    ax78.plot(times, laplacians[j+datasize], c='blue')



In [ ]:
# generate a dataset
T = 1000
datasize = 20
trials = 400
bw=20

# generate a random set of ramp parameters
beta = np.random.uniform(0, 4, datasize)
ln_sig = np.random.uniform(np.log(0.04), np.log(4), datasize)
ramp_params = np.column_stack((beta, np.exp(ln_sig)))

# generate a random set of step parameters
r = np.random.uniform(0.5, 6, datasize)
m = np.random.uniform(T/4, 3*T/4, datasize)
step_params = np.column_stack((m, r))

x_init = np.random.uniform(0, 0.5, datasize*2)


# generate simulations of each parameter set
ramp_dataset = np.array([models.RampModel(beta=ramp_params[i,0],
                                      sigma=ramp_params[i,1],
                                      x0 = x_init[i]).simulate(Ntrials=trials,
                                                               T=T)[0]
                                      for i in range(datasize)
                                      ]
                                    )
                                    
step_dataset = np.array([models.StepModel(m=step_params[j,0],
                                      r=step_params[j,1],
                                      x0 = x_init[j+datasize]).simulate(Ntrials=trials,
                                                                   T=T)[0]
                                      for j in range(datasize)
                                      ]
)

dataset = np.vstack((ramp_dataset, step_dataset))